<a href="https://colab.research.google.com/github/datasb44-bit/Greenwashing-detection/blob/main/Greenwashing_detection_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Greenwashing Detection Pipeline
# Outputs are saved to OUT_DIR.
# ============================================================

import os
import json
import re
from io import BytesIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import requests
from PIL import Image
from sklearn.cluster import KMeans

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    precision_recall_curve,
    ConfusionMatrixDisplay
)


try:
    import seaborn as sns
except Exception:
    sns = None

try:
    import nltk
    from nltk.sentiment import SentimentIntensityAnalyzer
except Exception:
    nltk = None
    SentimentIntensityAnalyzer = None

try:
    from wordcloud import WordCloud
except Exception:
    WordCloud = None

try:
    from scipy.stats import ttest_ind
except Exception:
    ttest_ind = None


# ---------------------------
# 0) Configuration
# ---------------------------
CSV_PATH = "/content/amazon.csv"  # Kaggle example: "/kaggle/input/<dataset>/<file>.csv"
OUT_DIR = "/content/amazon_repro_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_SEED = 42
TEST_SIZE = 0.20

# Section 3.4 image module.
IMG_SIZE = (100, 100)
K = 3
KMEANS_N_INIT = 10
IMAGE_TIMEOUT = 12

GREEN_RULE_TEXT = "green if (G>R) & (G>B) & (G>100)"

RF_PARAMS = {
    "n_estimators": 100,
    "max_depth": 10,
    "random_state": RANDOM_SEED
}

FEATURES = [
    "actual_price_num",
    "discount_pct_num",
    "rating_num",
    "img_green_flag",
    "img_green_ratio"
]

# Cost settings for threshold selection.
C_FN = 5.0
C_FP = 1.0

run_config = {
    "CSV_PATH": CSV_PATH,
    "OUT_DIR": OUT_DIR,
    "RANDOM_SEED": RANDOM_SEED,
    "TEST_SIZE": TEST_SIZE,
    "IMG_SIZE": IMG_SIZE,
    "KMEANS_K": K,
    "KMEANS_N_INIT": KMEANS_N_INIT,
    "GREEN_RULE": GREEN_RULE_TEXT,
    "RF_PARAMS": RF_PARAMS,
    "FEATURES": FEATURES,
    "COSTS": {"C_FN": C_FN, "C_FP": C_FP},
    "LABEL_RULE": "y=1 if (text_claim OR image_claim) AND review_does_not_confirm"
}

with open(os.path.join(OUT_DIR, "run_config.json"), "w") as f:
    json.dump(run_config, f, indent=2)

print("Outputs will be saved to:", OUT_DIR)


# ---------------------------
# 1) Data loading
# ---------------------------
# ---------------------------
# Optional: load frozen features for exact reproducibility
# ---------------------------
FEATURE_DATASET_PATH = os.path.join(OUT_DIR, "amazon_greenwashing_features.csv")

if os.path.exists(FEATURE_DATASET_PATH):
    print("Loading frozen feature dataset for exact reproducibility.")
    df = pd.read_csv(FEATURE_DATASET_PATH)

    # When loading frozen features, you can skip all steps that compute:
    # - numeric cleaning
    # - lexicon flags
    # - image features
    # - weak labels
    # Because they already exist in the frozen file.

else:
    print("Frozen features not found. Running full pipeline from raw CSV and image URLs.")
    df = pd.read_csv(CSV_PATH)

#df = pd.read_csv(CSV_PATH)

needed_cols = [
    "about_product",
    "review_content",
    "img_link",
    "actual_price",
    "discount_percentage",
    "rating"
]
df = df[needed_cols].copy()

# Missing ratings are handled later.
df = df.dropna(subset=["about_product", "review_content", "img_link", "actual_price", "discount_percentage"]).copy()
df = df[df["review_content"].astype(str).str.strip() != ""].copy()


# ---------------------------
# 2) Numeric preprocessing
# ---------------------------
def parse_money(x):
    if pd.isna(x):
        return np.nan
    s = re.sub(r"[^\d.]", "", str(x))
    return float(s) if s else np.nan

def parse_percent(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace("%", "")
    s = re.sub(r"[^\d.]", "", s)
    return float(s) if s else np.nan

df["actual_price_num"] = df["actual_price"].apply(parse_money)
df["discount_pct_num"] = df["discount_percentage"].apply(parse_percent)

# Missing ratings are replaced by zero.
df["rating_num"] = pd.to_numeric(df["rating"], errors="coerce").fillna(0.0)

# Price and discount are core features.
df = df.dropna(subset=["actual_price_num", "discount_pct_num"]).copy()


# ---------------------------
# 3) Lexicon-based text detection
# ---------------------------
explicit_terms = [
    # Direct "green" / environmental claims
    "eco-friendly", "eco friendly", "environmentally friendly", "environmental friendly",
    "eco-conscious", "eco conscious", "earth-friendly", "earth friendly",
    "planet-friendly", "planet friendly",

    # Sustainability framing
    "sustainable", "sustainability", "environmentally sustainable",

    # End-of-life / materials
    "biodegradable", "compostable", "recyclable", "recycled", "recycled material",
    "recycled materials", "plastic-free", "bio-based",

    # Carbon claims
    "carbon neutral", "carbon-neutral", "carbon footprint", "carbon offset",
    "carbon-offset", "low carbon", "low-carbon",

    # Waste claims
    "zero waste", "zero-waste",

    # Energy source claims
    "renewable", "renewable energy", "clean energy", "green energy",

    # Packaging claims
    "sustainable packaging", "eco-friendly packaging", "eco friendly packaging", "recyclable packaging",

    # Social/ethical sustainability
    "ethical", "ethical sourcing", "fair trade", "fair-trade"
]

implicit_terms = [
    # Efficiency cues
    "energy saving", "energy-saving", "energy efficient", "energy-efficient",
    "low energy", "low-energy", "resource efficient", "resource-efficient",
    "resource saving", "resource-saving",

    # Emissions/impact cues
    "reduced footprint", "low emissions", "low-emission", "low-emissions", "reduced emissions",

    # Durability / longevity cues
    "durable", "long lasting", "long-lasting", "long life", "long-life",
    "long-lasting product", "long lasting product",

    # Waste reduction cues
    "waste reducing", "waste-reducing", "minimal waste",

    # General impact & "Nature" wording
    "environmental impact", "reduced impact", "natural", "all-natural", "nature-inspired"
]

GREEN_TERMS = list(dict.fromkeys([t.lower() for t in (explicit_terms + implicit_terms)]))

def has_green_terms(text):
    if pd.isna(text):
        return 0
    t = str(text).lower()
    return int(any(term in t for term in GREEN_TERMS))

df["has_green_claim_text"] = df["about_product"].apply(has_green_terms)
df["review_mentions_green"] = df["review_content"].apply(has_green_terms)


# ---------------------------
# 4) Image-based green cues (Section 3.4)
# ---------------------------
_img_cache = {}

def fetch_image(url, timeout=IMAGE_TIMEOUT):
    if pd.isna(url) or not isinstance(url, str) or not url.startswith("http"):
        return None
    if url in _img_cache:
        return _img_cache[url]
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        r = requests.get(url, headers=headers, timeout=timeout)
        r.raise_for_status()
        img = Image.open(BytesIO(r.content)).convert("RGB")
        _img_cache[url] = img
        return img
    except Exception:
        _img_cache[url] = None
        return None

def image_green_features(url):
    img = fetch_image(url)
    if img is None:
        return 0, 0.0

    img = img.resize(IMG_SIZE)
    arr = np.asarray(img, dtype=np.float32)
    flat = arr.reshape(-1, 3)

    R = flat[:, 0]
    G = flat[:, 1]
    B = flat[:, 2]

    green_pixels = (G > R) & (G > B) & (G > 100.0)
    img_green_ratio = float(green_pixels.mean())

    try:
        km = KMeans(n_clusters=K, n_init=KMEANS_N_INIT, random_state=RANDOM_SEED)
        km.fit(flat)
        centers = km.cluster_centers_

        Rc = centers[:, 0]
        Gc = centers[:, 1]
        Bc = centers[:, 2]

        img_green_flag = int(np.any((Gc > Rc) & (Gc > Bc) & (Gc > 100.0)))
    except Exception:
        img_green_flag = 0

    return img_green_flag, img_green_ratio

img_feats = df["img_link"].apply(image_green_features)
df["img_green_flag"] = img_feats.apply(lambda x: x[0])
df["img_green_ratio"] = img_feats.apply(lambda x: x[1])


# ---------------------------
# 5) Weak label construction
# ---------------------------
df["has_green_claim_any"] = ((df["has_green_claim_text"] == 1) | (df["img_green_flag"] == 1)).astype(int)
df["y_greenwashing"] = ((df["has_green_claim_any"] == 1) & (df["review_mentions_green"] == 0)).astype(int)


# ---------------------------
# 6) Save frozen and processed datasets
# ---------------------------
FEATURE_DATASET_PATH = os.path.join(OUT_DIR, "amazon_greenwashing_features.csv")
PROCESSED_PATH = os.path.join(OUT_DIR, "amazon_processed_with_labels.csv")

keep_cols = [
    "about_product", "review_content", "img_link",
    "actual_price", "discount_percentage", "rating",
    "actual_price_num", "discount_pct_num", "rating_num",
    "has_green_claim_text", "review_mentions_green",
    "img_green_flag", "img_green_ratio",
    "has_green_claim_any", "y_greenwashing"
]

df[keep_cols].to_csv(FEATURE_DATASET_PATH, index=False)
df.to_csv(PROCESSED_PATH, index=False)

print("Saved frozen feature dataset:", FEATURE_DATASET_PATH)
print("Saved processed dataset:", PROCESSED_PATH)


# ---------------------------
# 7) Model training
# ---------------------------
X = df[FEATURES].copy()
y = df["y_greenwashing"].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=y
)

np.save(os.path.join(OUT_DIR, "train_idx.npy"), X_train.index.to_numpy())
np.save(os.path.join(OUT_DIR, "test_idx.npy"), X_test.index.to_numpy())

clf = RandomForestClassifier(**RF_PARAMS)
clf.fit(X_train, y_train)


# ---------------------------
# 8) Evaluation at threshold 0.5
# ---------------------------
y_pred = clf.predict(X_test)
proba = clf.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred)
print("=== Confusion Matrix ===")
print(cm)

report_dict = classification_report(y_test, y_pred, digits=3, zero_division=0, output_dict=True)
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, digits=3, zero_division=0))

print("Threshold=0.5 precision:", precision_score(y_test, (proba >= 0.5).astype(int), zero_division=0))
print("Threshold=0.5 recall   :", recall_score(y_test, (proba >= 0.5).astype(int), zero_division=0))

# Save outputs as tables.
cm_df = pd.DataFrame(cm, index=["true_0", "true_1"], columns=["pred_0", "pred_1"])
cm_df.to_csv(os.path.join(OUT_DIR, "confusion_matrix.csv"), index=True)

report_df = pd.DataFrame(report_dict).T
report_df.to_csv(os.path.join(OUT_DIR, "classification_report.csv"), index=True)

pred_df = pd.DataFrame({
    "row_index": X_test.index,
    "y_true": y_test.values,
    "y_pred": y_pred,
    "proba_greenwashing": proba
})
pred_df.to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)


# ---------------------------
# 9) Threshold sweep table
# ---------------------------
def metrics_at_threshold(t):
    pred = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0

    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    beta = 2.0
    f2 = ((1 + beta**2) * precision * recall / ((beta**2) * precision + recall)) if ((beta**2) * precision + recall) else 0.0

    return precision, recall, f1, f2, tn, fp, fn, tp

thr_candidates = np.unique(proba)
rows = []
for t in thr_candidates:
    precision, recall, f1, f2, tn, fp, fn, tp = metrics_at_threshold(t)
    rows.append([t, precision, recall, f1, f2, tn, fp, fn, tp])

thr_table = pd.DataFrame(rows, columns=["threshold","precision","recall","F1","F2","TN","FP","FN","TP"])
thr_table["expected_cost"] = C_FN * thr_table["FN"] + C_FP * thr_table["FP"]

thr_table.to_csv(os.path.join(OUT_DIR, "threshold_sweep_table.csv"), index=False)

best_f1 = thr_table.loc[thr_table["F1"].idxmax()].copy()
best_f2 = thr_table.loc[thr_table["F2"].idxmax()].copy()
best_cost = thr_table.loc[thr_table["expected_cost"].idxmin()].copy()

def nearest_row(val):
    return thr_table.iloc[(thr_table["threshold"] - val).abs().argsort()[:1]].iloc[0].copy()

row_05 = nearest_row(0.5)

summary = pd.DataFrame([
    {"Selection":"Default ~0.5", **row_05[["threshold","precision","recall","F1","F2","FP","FN","TP","expected_cost"]].to_dict()},
    {"Selection":"Max F1", **best_f1[["threshold","precision","recall","F1","F2","FP","FN","TP","expected_cost"]].to_dict()},
    {"Selection":"Max F2", **best_f2[["threshold","precision","recall","F1","F2","FP","FN","TP","expected_cost"]].to_dict()},
    {"Selection":f"Min expected cost (C_FN={C_FN:g}, C_FP={C_FP:g})", **best_cost[["threshold","precision","recall","F1","F2","FP","FN","TP","expected_cost"]].to_dict()},
])

summary_rounded = summary.copy()
for col in ["threshold","precision","recall","F1","F2"]:
    summary_rounded[col] = summary_rounded[col].astype(float).round(3)
summary_rounded["expected_cost"] = summary_rounded["expected_cost"].astype(float).round(1)

summary_rounded.to_csv(os.path.join(OUT_DIR, "threshold_selection_summary.csv"), index=False)

print("\n=== Threshold selection summary ===")
print(summary_rounded.to_string(index=False))


# ---------------------------
# 10) Precision-recall charts (save)
# ---------------------------
precisions, recalls, thresholds = precision_recall_curve(y_test, proba)
prec_t = precisions[1:]
rec_t = recalls[1:]
thr = thresholds

# (A) Precision and recall vs threshold.
plt.figure(figsize=(8, 5))
plt.plot(thr, prec_t, label="Precision")
plt.plot(thr, rec_t, label="Recall")
plt.axvline(0.5, linestyle="--", label="Threshold=0.5")
plt.xlabel("Decision threshold")
plt.ylabel("Score")
plt.title("Precision and Recall versus Threshold (Random Forest)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "precision_recall_vs_threshold.png"), dpi=200)
plt.show()

# (B) Precision-recall curve.
plt.figure(figsize=(6, 5))
plt.plot(recalls, precisions)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "precision_recall_curve.png"), dpi=200)
plt.show()

# (C) Marked operating points.
t_f1 = float(best_f1["threshold"])
t_f2 = float(best_f2["threshold"])
t_cost = float(best_cost["threshold"])

plt.figure(figsize=(9, 5.5))
plt.plot(thr, prec_t, label="Precision")
plt.plot(thr, rec_t, label="Recall")
plt.axvline(0.5, linestyle=":", label="Reference ~0.5")
plt.axvline(t_f1, linestyle="--", label=f"Max F1 t={t_f1:.3f}")
plt.axvline(t_f2, linestyle="--", label=f"Max F2 t={t_f2:.3f}")
if (abs(t_cost - t_f1) > 1e-6) and (abs(t_cost - t_f2) > 1e-6):
    plt.axvline(t_cost, linestyle="--", label=f"Min cost t={t_cost:.3f}")

plt.xlabel("Decision threshold")
plt.ylabel("Score")
plt.title("Precision and Recall versus Threshold (Selected Operating Points)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "precision_recall_threshold_marked.png"), dpi=200)
plt.show()


# ---------------------------
# A) VADER sentiment scores
# ---------------------------
if nltk is not None and SentimentIntensityAnalyzer is not None:
    try:
        nltk.download("vader_lexicon", quiet=True)
        sia = SentimentIntensityAnalyzer()

        def vader_compound(text):
            if pd.isna(text):
                return np.nan
            return sia.polarity_scores(str(text))["compound"]

        df["sentiment_score"] = df["review_content"].apply(vader_compound)

        def sentiment_label(s):
            if pd.isna(s):
                return np.nan
            if s >= 0.05:
                return "Positive"
            if s <= -0.05:
                return "Negative"
            return "Neutral"

        df["sentiment_category"] = df["sentiment_score"].apply(sentiment_label)
    except Exception:
        df["sentiment_score"] = np.nan
        df["sentiment_category"] = np.nan
else:
    df["sentiment_score"] = np.nan
    df["sentiment_category"] = np.nan

# Review sustainability indicator.
df["eco_conscious_review"] = df["review_mentions_green"].astype(int)


# ---------------------------
# KDE sentiment distribution
# ---------------------------
if sns is not None:
    plt.figure(figsize=(8,5))
    sns.kdeplot(df.loc[df["eco_conscious_review"]==1, "sentiment_score"].dropna(),
                label="With sustainability mention", fill=True)
    sns.kdeplot(df.loc[df["eco_conscious_review"]==0, "sentiment_score"].dropna(),
                label="Without sustainability mention", fill=True)
    plt.title("Sentiment Score Distribution for Reviews")
    plt.xlabel("VADER compound sentiment")
    plt.ylabel("Density")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "sentiment_kde.png"), dpi=200)
    plt.show()

